# Train the ASL alphabet recognizer

Fine-tunes MediaPipe's pretrained gesture recognizer on the public
[ASL Alphabet](https://www.kaggle.com/datasets/grassknoted/asl-alphabet) dataset.

**How to use:** open in Google Colab → *Runtime → Run all* → wait ~15–25 min →
the last cell downloads `gesture_recognizer.task`. Copy it to
`app/android/app/src/main/assets/gesture_recognizer.task` and rebuild the app.

Labels produced: `A`–`Z`, `space`, `del`, and `none` (no sign).
J and Z involve motion in real ASL; here they're learned from the dataset's still frames, so expect them to be weak.

In [ ]:
# Model Maker pins its own TensorFlow version; this can take a few minutes.
!pip install -q mediapipe-model-maker kagglehub

## 1. Download data

In [ ]:
import glob, os, random, shutil, zipfile, urllib.request
import kagglehub

asl_root = kagglehub.dataset_download("grassknoted/asl-alphabet")
# Find the training folder (the one that contains A/ and del/ sub-folders).
train_dir = next(os.path.dirname(p) for p in glob.glob(f"{asl_root}/**/A", recursive=True)
                 if os.path.isdir(os.path.join(os.path.dirname(p), "del")))
print("ASL training images:", train_dir)

# The ASL dataset's "nothing" class has no hands in it, so MediaPipe would skip every image.
# Use the "none" class from MediaPipe's sample data instead: real hands in random poses.
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-tasks/gesture_recognizer/rps_data_sample.zip", "rps.zip")
zipfile.ZipFile("rps.zip").extractall("rps")
none_dir = glob.glob("rps/**/none", recursive=True)[0]

## 2. Build a smaller dataset
The full set is 87k images; ~400 per letter is plenty and keeps hand detection quick.

In [ ]:
IMAGES_PER_CLASS = 400
LABELS = [chr(c) for c in range(ord("A"), ord("Z") + 1)] + ["space", "del"]
DATA = "dataset"
random.seed(0)
shutil.rmtree(DATA, ignore_errors=True)

for label in LABELS:
    files = sorted(glob.glob(os.path.join(train_dir, label, "*.jpg")))
    os.makedirs(os.path.join(DATA, label))
    for f in random.sample(files, min(IMAGES_PER_CLASS, len(files))):
        shutil.copy(f, os.path.join(DATA, label))

shutil.copytree(none_dir, os.path.join(DATA, "none"))
for d in sorted(os.listdir(DATA)):
    print(d, len(os.listdir(os.path.join(DATA, d))))

## 3. Train

In [ ]:
from mediapipe_model_maker import gesture_recognizer

data = gesture_recognizer.Dataset.from_folder(
    dirname=DATA, hparams=gesture_recognizer.HandDataPreprocessingParams())
train_data, rest = data.split(0.8)
val_data, test_data = rest.split(0.5)

options = gesture_recognizer.GestureRecognizerOptions(
    hparams=gesture_recognizer.HParams(export_dir="exported_model", epochs=30))
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data, validation_data=val_data, options=options)

## 4. Evaluate and export

In [ ]:
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Test accuracy: {acc:.1%}")
model.export_model()

In [ ]:
from google.colab import files
files.download("exported_model/gesture_recognizer.task")